# 04 · Plotting with gog4j

Numbers became a table; now the table becomes a picture. **gog4j** is ggplot2
for JavaFX. The grammar is three pieces:

* **data** — here a `HardwoodTable`, read straight from the Parquet
* **aes** — which columns drive x, y, colour, …
* **geoms** — how to draw them (points, lines, bars, density, …)

Build a plot with `Ggplot.ggplot(...)`, add layers with `.geoms(...)`, then
either `display(...)` it or export it to SVG.

In [0]:
addDependency("org.jtaccuino:gog4j:0.5-SNAPSHOT")
addDependency("org.jtaccuino:gog4j-hardwood:0.5-SNAPSHOT")

import org.jtaccuino.gog.*;
import org.jtaccuino.gog.labs.Labs;
import org.jtaccuino.gog.render.SvgExporter;
import org.jtaccuino.gog.theme.Theme;
import org.jtaccuino.gog.hardwood.HardwoodTable;
import java.nio.file.Files;

var full = HardwoodTable.ofFile(cwd.resolve("data/celestrak_gp_catalog.parquet"));

// Project to just the fields this module plots. Reading and keeping only the
// columns you need is the columnar habit; here it also keeps the frame small.
var table = HardwoodTable.ofColumns(java.util.Map.of(
        "inclination", full.column("inclination"),
        "eccentricity", full.column("eccentricity"),
        "period_min", full.column("period_min"),
        "apogee_km", full.column("apogee_km"),
        "orbit_class", full.column("orbit_class"),
        "satcat_object_type", full.column("satcat_object_type")));

void save(GgFigure figure, String name) {
    try {
        var path = cwd.resolve(name);
        new SvgExporter().size(1200, 800).batchPoints(true).write(figure, path);
        println("saved %s (%s bytes)", name, String.format(java.util.Locale.ROOT, "%,d", Files.size(path)));
    } catch (Exception e) {
        println("could not save %s: %s", name, e);
    }
}

## 1 · Data + aes + geoms

One point per satellite: inclination across, apogee up, coloured by orbit class.
That single `color(...)` is the whole point of a grammar of graphics.

In [0]:
// ── TODO 1 ────────────────────────────────────────────────────────────
// Build the scatter plot into `p`:
//   data  table
//   aes   x = inclination, y = apogee_km, colour = orbit_class
//   geom  points

// Plot<HardwoodTable> p = Ggplot.ggplot(table,
//         Aes.aes().x("inclination").y("apogee_km").color("orbit_class"))
//     .geoms(Geoms.point());

In [0]:
// ── given ── look at it, and save a copy
display(p);
save(p, "01-scatter.svg");

## 2 · Labels

A plot without axis labels is a riddle. `Labs.labs(...)` answers it.

In [0]:
// ── TODO 2 ────────────────────────────────────────────────────────────
// Add a title and axis labels.

// p = p.labs(Labs.labs("Everything in orbit", "Inclination (deg)", "Apogee (km)"));

In [0]:
display(p);
save(p, "02-labelled.svg");

## 3 · Theme

A theme changes every colour at once. On a projector, dark wins.

In [0]:
// ── TODO 3 ────────────────────────────────────────────────────────────
// Switch to the dark theme.

// p = p.theme(Theme.theme_dark());

In [0]:
display(p);
save(p, "03-dark.svg");

## 4 · Facets

Faceting splits one plot into a small multiple per category — the fastest way to
see that the classes are genuinely different populations.

In [0]:
// ── TODO 4 ────────────────────────────────────────────────────────────
// Split the plot into one panel per orbit class, in a 2-wide grid.

// p = p.facets(Facets.wrap("orbit_class", 2));

In [0]:
display(p);
save(p, "04-facets.svg");

## 5 · Reference lines

The geostationary belt sits at 35 786 km. `Geoms.hline(...)` draws it, and one
glance then explains the horizontal stripe of GEO objects.

In [0]:
// ── TODO 5 ────────────────────────────────────────────────────────────
// Add a horizontal line at the geostationary altitude, keeping the points.

// p = p.geoms(Geoms.point(), Geoms.hline(35786));

In [0]:
display(p);
save(p, "05-geobelt.svg");

## 6 · Export

`display` is for you; `SvgExporter` is for the paper. Export the finished figure
at print size, with the 21 000 points batched so the file stays editable.

In [0]:
// ── TODO 6 ────────────────────────────────────────────────────────────
// Write the final plot to "celestrak-orbits.svg" at 1600 x 1000.

// new SvgExporter().size(1600, 1000).batchPoints(true)
//     .write(p, cwd.resolve("celestrak-orbits.svg"));
// println("wrote celestrak-orbits.svg");

## The bridge to machine learning

Everything so far *described* the catalog. Now predict something from it. Can the
orbital elements alone tell debris from a working satellite?

`satcat_object_type` gives the ground truth: `PAY` (payload) or `DEB` / `R/B`
(debris). The features are the elements — inclination, eccentricity, period, mean
motion, drag. We stop at the train/test split: **this is where Zoran takes over**
and puts a network on top.

In [0]:
// ── given ── turn it into a supervised problem
addDependency("org.dflib:dflib:2.0.0-M7")
addDependency("org.dflib:dflib-parquet:2.0.0-M7")
import org.dflib.DataFrame;
import org.dflib.parquet.Parquet;
import static org.dflib.Exp.*;

var df = Parquet.loader().load(cwd.resolve("data/celestrak_gp_catalog.parquet"));

// keep only rows with a usable label
var labelled = df.rows(
        $str("satcat_object_type").eq("PAY")
            .or($str("satcat_object_type").eq("DEB"))
            .or($str("satcat_object_type").eq("R/B")))
    .select();

println("labelled %s of %s rows", String.format(java.util.Locale.ROOT, "%,d", labelled.height()),
        String.format(java.util.Locale.ROOT, "%,d", df.height()));
System.out.println(labelled.group("satcat_object_type").agg($col("satcat_object_type"), count()));

In [0]:
// ── given ── features and a deterministic split
var features = labelled
    .cols("inclination", "eccentricity", "period_min", "mean_motion", "bstar")
    .select();

int split = (int) (features.height() * 0.8);
var train = features.rowsRange(0, split).select();
var test  = features.rowsRange(split, features.height()).select();

println("train %s   test %s   features %d",
        String.format(java.util.Locale.ROOT, "%,d", train.height()),
        String.format(java.util.Locale.ROOT, "%,d", test.height()),
        features.width());
println("→ hand `train` and `test` (with their labels) to the ML module.");

## ★ BONUS 1 — Pairs and composition

Two dozen features at once, as a scatterplot matrix. `matrixPlot` does the grid;
`composedPlot` arranges finished figures side by side.

In [0]:
// ── ★ BONUS 1 ─────────────────────────────────────────────────────────
var pairs = Ggplot.matrixPlot(table,
    Aes.aes().color("orbit_class"),
    "inclination", "eccentricity", "period_min", "apogee_km");
display(pairs);
save(pairs, "bonus-pairs.svg");

## ★ BONUS 2 — A three-dimensional orbit cloud

Add inclination, period and eccentricity as x/y/z and let the points fill the
volume.

In [0]:
// ── ★ BONUS 2 ─────────────────────────────────────────────────────────
var cloud = Ggplot.ggplot3d(table,
        Aes.aes().x("inclination").y("period_min").z("eccentricity").color("orbit_class"))
    .geoms(Geoms.point3d());
display(cloud);
save(cloud, "bonus-3d.svg");

## ★ BONUS 3 — The wall chart

A filled 2-D density instead of 21 000 points, with the GEO belt drawn on top.
This is the figure that goes on the poster.

In [0]:
// ── ★ BONUS 3 ─────────────────────────────────────────────────────────
var wall = Ggplot.ggplot(table,
        Aes.aes().x("inclination").y("apogee_km").fill("orbit_class"))
    .geoms(Geoms.density2dFilled(), Geoms.hline(35786))
    .labs(Labs.labs("Where satellites live", "Inclination (deg)", "Apogee (km)"))
    .theme(Theme.theme_dark());
display(wall);
save(wall, "bonus-wallchart.svg");

That is the whole pipeline: **Hardwood** read it, **dflib** shaped it,
**gog4j** drew it, and you left a labelled train/test split on the table for the
model.

Back to the [overview](../narrative/00-overview.md).